> **CO2 fork (2026-08-18)** — created by Hermes on branch `feat/co2-adapters` from `DPO_train_test/Base_run/CO2_train_multi_country.ipynb` @ `eb226a5` (origin/main).
> Ksennia's original notebooks are untouched. Country set swapped to IND, IDN, NGA, EGY, TUR, NLD, BRA, GRC; Drive root → `MyDrive/DPO_CO2/`.
> Hyperparameters identical to Base_run: BETA=0.1, MAX_LENGTH=768, batch 1 x accum 16, 1 epoch, LR 1e-4 cosine, LoRA r=16 alpha=32 dropout=0.05 on q/k/v/o, fp16, QLoRA NF4+double-quant.


# CO2 DPO training — original datasets (fork of Base_run)

This notebook trains one adapter per country from the files produced by `DPO_data_preparation_multi_country.ipynb`.

Prepared input naming:
- `USA_train_with_ref.jsonl`
- `MEX_train_with_ref.jsonl`
- etc.

Adapters are saved separately under `dpo_qlora_adapters/<COUNTRY_TAG>/`.


In [1]:
!pip -q install -U "transformers>=4.41.0" "datasets>=2.18.0" "accelerate>=0.30.0" \
                 "trl>=0.11.0" "peft>=0.11.1" "bitsandbytes>=0.46.1" "safetensors>=0.4.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import DPOTrainer, DPOConfig

# Hugging Face login is needed for gated Llama weights.
from huggingface_hub import notebook_login
notebook_login()

In [9]:
from pathlib import Path

# Train one or more countries sequentially.
# US maps to data files beginning with USA_, but its adapter folder is US/.
RUN_COUNTRIES = ['IND', 'IDN', 'NGA', 'EGY', 'TUR', 'NLD', 'BRA', 'GRC']

COUNTRY_NAME_MAP = {
    "US": "USA",
}

def resolve_country_codes(country_code):
    data_country_code = COUNTRY_NAME_MAP.get(country_code, country_code)
    adapter_tag = country_code
    return data_country_code, adapter_tag

# Change data directory
DATA_DIR = Path("/content/drive/MyDrive/DPO_CO2")
ADAPTER_ROOT = DATA_DIR / "dpo_qlora_adapters"
ADAPTER_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"


def build_training_paths(country_code):
    data_country_code, adapter_tag = resolve_country_codes(country_code)
    data_file = DATA_DIR / f"{data_country_code}_train_with_ref.jsonl"
    output_dir = ADAPTER_ROOT / adapter_tag
    return data_file, output_dir, data_country_code, adapter_tag

for country_code in RUN_COUNTRIES:
    print(country_code, "->", build_training_paths(country_code))

RUS -> (PosixPath('/content/drive/MyDrive/DPO/RUS_train_with_ref.jsonl'), PosixPath('/content/drive/MyDrive/DPO/dpo_qlora_adapters/RUS'), 'RUS', 'RUS')


In [10]:
import os

# Set before importing torch/transformers/accelerate.
os.environ["ACCELERATE_MIXED_PRECISION"] = "fp16"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TRANSFORMERS_NO_BF16"] = "1"

import torch

torch.set_default_dtype(torch.float16)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU for 4-bit QLoRA training.")

CUDA: 12.8
GPU: Tesla T4


In [11]:
def build_user_prompt(prompt_text: str) -> str:
    return (
        "You are answering a questionnaire as an individual person. "
        "Respond naturally and thoughtfully, as someone would in real life. "
        "Do not mention being an AI or assistant. "
        "Keep the answer short, under 3 sentences. "
        "Give a sincere, human-like answer.\n\n"
        "Situation:\n"
        f"{prompt_text.strip()}\n\n"
        "Answer:"
    )


def format_prompt_text(tokenizer, prompt_text: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": build_user_prompt(prompt_text)}],
        tokenize=False,
        add_generation_prompt=True,
    )


def format_dataset_example(ex, tokenizer):
    ex["prompt"] = format_prompt_text(tokenizer, ex["prompt"])
    return ex


def validate_training_dataset(dataset, name):
    required = {"prompt", "chosen", "rejected", "ref_chosen_logps", "ref_rejected_logps"}
    missing = required - set(dataset.column_names)
    if missing:
        raise ValueError(f"{name} is missing required columns: {sorted(missing)}")

In [14]:
# Training settings kept close to the original notebook.
BETA = 0.1
MAX_LENGTH = 768
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
NUM_TRAIN_EPOCHS = 1
LEARNING_RATE = 1e-4
SAVE_STEPS = 5
SAVE_TOTAL_LIMIT = 2


def train_country(country_code):
    data_file, output_dir, data_country_code, adapter_tag = build_training_paths(country_code)

    if not data_file.exists():
        raise FileNotFoundError(
            f"Prepared training file not found: {data_file}. "
            "Run the data-preparation notebook for this country first."
        )

    print(f"\n===== Training {country_code} ({data_country_code}) =====")
    print("Input:", data_file)
    print("Adapter output:", output_dir)

    free, total = torch.cuda.mem_get_info()
    print(f"Free VRAM: {free/1024**3:.2f} GiB / Total: {total/1024**3:.2f} GiB")

    compute_dtype = torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map={"": 0},
        low_cpu_mem_usage=True,
        torch_dtype=torch.float16,
    )

    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()

    ds = load_dataset("json", data_files={"train": str(data_file)})
    train_dataset = ds["train"]
    validate_training_dataset(train_dataset, str(data_file))

    # Use exactly the same prompt/chat formatting that was used to compute ref log-probs.
    train_dataset = train_dataset.map(
        lambda ex: format_dataset_example(ex, tokenizer)
    )

    peft_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, peft_cfg)

    # Preserve the dtype handling from the original working notebook.
    for _, p in model.named_parameters():
        if p.requires_grad:
            p.data = p.data.to(torch.float16)

    model.print_trainable_parameters()

    dpo_args = DPOConfig(
        output_dir=str(output_dir),
        beta=BETA,
        max_length=MAX_LENGTH,
        truncation_mode="keep_end",
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        #warmup_ratio=0.03,
        logging_steps=10,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        report_to="none",
        fp16=True,
        bf16=False,
    )

    trainer = DPOTrainer(
        model=model,
        ref_model=None,
        args=dpo_args,
        train_dataset=train_dataset,
        processing_class=tokenizer,
    )

    # Original notebook forced trainable parameters to fp32 after trainer creation.
    for _, p in trainer.model.named_parameters():
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.to(torch.float32)

    trainable = next(p for p in trainer.model.parameters() if p.requires_grad)
    print("Trainable dtype after trainer init:", trainable.dtype)
    print("Accelerate mixed_precision:", trainer.accelerator.mixed_precision)

    trainer.train()

    # Save adapter-only weights and tokenizer in the country-specific folder.
    output_dir.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    print("Saved adapter-only model to:", output_dir)

    # Free VRAM before the next country.
    del trainer, model, train_dataset, ds
    torch.cuda.empty_cache()

    return output_dir

In [15]:
# Train all requested countries sequentially.
saved_adapters = {}

for country_code in RUN_COUNTRIES:
    saved_adapters[country_code] = train_country(country_code)

print("\nSaved adapters:")
for country_code, path in saved_adapters.items():
    print(f"  {country_code}: {path}")


===== Training RUS (RUS) =====
Input: /content/drive/MyDrive/DPO/RUS_train_with_ref.jsonl
Adapter output: /content/drive/MyDrive/DPO/dpo_qlora_adapters/RUS
Free VRAM: 8.95 GiB / Total: 14.56 GiB


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695


/tmp/ipykernel_1405/1203468180.py:78: FutureWarning: The `'keep_end'` truncation mode is deprecated and will be removed in v2.0.0. Use `truncation_mode='keep_start'` (the default) instead.
  dpo_args = DPOConfig(


Adding EOS to train dataset:   0%|          | 0/526 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/526 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Trainable dtype after trainer init: torch.float32
Accelerate mixed_precision: fp16


Step,Training Loss
10,0.398242
20,0.075293
30,0.080225


Saved adapter-only model to: /content/drive/MyDrive/DPO/dpo_qlora_adapters/RUS

Saved adapters:
  RUS: /content/drive/MyDrive/DPO/dpo_qlora_adapters/RUS
